In [ ]:
import pandas as pd
from tabpfn import TabPFNClassifier, TabPFNRegressor
from tabpfn_extensions.embedding import TabPFNEmbedding
import numpy as np
from dotenv import load_dotenv
from sklearn.model_selection import StratifiedShuffleSplit
load_dotenv(".env")

In [2]:
from benchmark_datasets import OpenMLBenchmark

In [ ]:
bench = OpenMLBenchmark()
print("Benchmark suite (20 datasets):")
print(bench.list().to_string(index=False))

ds = bench.load("wdbc")
print(f"\nLoaded {ds.name!r}: X={ds.X.shape} ({ds.X.dtype}), y={ds.y.shape}")
print(f"classes: {ds.n_classes}, feature count: {len(ds.feature_names)}")

In [9]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier

In [14]:
from sklearn.metrics import accuracy_score
sss = StratifiedShuffleSplit(1,test_size=0.8)
for i, (train_index, test_index) in enumerate(sss.split(ds.X, ds.y)):
    X_train = ds.X[train_index]
    y_train = ds.y[train_index]
    X_test = ds.X[test_index]
    y_test = ds.y[test_index]

    clf = TabPFNClassifier()
    clf.fit(X_train, y_train)
    base_pfn_pred = clf.predict(X_test)
    print('Base tabPFN acc: ', accuracy_score(y_test,base_pfn_pred))

    X_aug = X_train
    y_aug = y_train
    for _ in range(3):
        aug_size = round(len(X_train)*1)
        rand_mask = np.random.rand(aug_size, len(X_train[0])) > 0.95
        rep_val = np.random.rand(aug_size, len(X_train[0])) > 0.5
        X_aug_new = X_train * (1-rand_mask) + rep_val*rand_mask

        y_aug_new = clf.predict(X_aug_new)

        X_aug = np.concat([X_aug, X_aug_new])
        y_aug = np.concat([y_aug, y_aug_new])
    
    for n_estim in [1,2,3,5,10,100]:
        rfr = RandomForestClassifier(n_estimators=n_estim)
        rfr.fit(X_train, y_train)
        base_rfr_pred = rfr.predict(X_test)
        print(F'RFR (n_estim={n_estim}) base acc: ', accuracy_score(y_test,base_rfr_pred))

        rfr_aug = RandomForestClassifier(n_estimators=n_estim)
        rfr_aug.fit(X_aug, y_aug)
        rfr_aug_pred = rfr_aug.predict(X_test)

        print(F'RFR (n_estim={n_estim}) aug acc: ', accuracy_score(y_test,rfr_aug_pred))

    for hid_size in [(8,),(32,),(128,), (16,16,)]:
        mlp = MLPClassifier(hidden_layer_sizes=n_estim, max_iter=1000)
        mlp.fit(X_train, y_train)
        base_mlp_pred = mlp.predict(X_test)
        print(f'MLP (hidden_l={hid_size}) base acc: ', accuracy_score(y_test,base_mlp_pred))

        mlp_aug = MLPClassifier(hidden_layer_sizes=n_estim, max_iter=1000)
        mlp_aug.fit(X_aug, y_aug)
        mlp_aug_pred= mlp_aug.predict(X_test)

        print(f'MLP (hidden_l={hid_size}) aug acc: ', accuracy_score(y_test,mlp_aug_pred))


Base tabPFN acc:  0.9692982456140351
RFR (n_estim=1) base acc:  0.9078947368421053
RFR (n_estim=1) aug acc:  0.9035087719298246
RFR (n_estim=2) base acc:  0.875
RFR (n_estim=2) aug acc:  0.9298245614035088
RFR (n_estim=3) base acc:  0.9517543859649122
RFR (n_estim=3) aug acc:  0.9035087719298246
RFR (n_estim=5) base acc:  0.918859649122807
RFR (n_estim=5) aug acc:  0.9385964912280702
RFR (n_estim=10) base acc:  0.9407894736842105
RFR (n_estim=10) aug acc:  0.9495614035087719
RFR (n_estim=100) base acc:  0.9429824561403509
RFR (n_estim=100) aug acc:  0.9407894736842105
MLP (hidden_l=(8,)) base acc:  0.36403508771929827
MLP (hidden_l=(8,)) aug acc:  0.9210526315789473
MLP (hidden_l=(32,)) base acc:  0.37280701754385964
MLP (hidden_l=(32,)) aug acc:  0.9276315789473685
MLP (hidden_l=(128,)) base acc:  0.3793859649122807
MLP (hidden_l=(128,)) aug acc:  0.918859649122807
MLP (hidden_l=(16, 16)) base acc:  0.5964912280701754
MLP (hidden_l=(16, 16)) aug acc:  0.9232456140350878
